In [3]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import pathlib

from architector import io_ptable,convert_io_molecule,view_structures

In [13]:
from bokeh.models import (
    ColumnDataSource,
    LinearColorMapper,
    LogColorMapper,
    ColorBar,
    BasicTicker,
    LogTicker
)
from bokeh.plotting import figure, output_file
from bokeh.io import show as show_
from bokeh.sampledata.periodic_table import elements
from bokeh.transform import dodge
from csv import reader
from matplotlib.colors import Normalize, LogNorm, to_hex
from matplotlib.cm import (
    plasma,
    inferno,
    magma,
    viridis,
    cividis,
    turbo,
    ScalarMappable,
)
from pandas import options
from typing import List
import warnings


def ptable_plotter(
    filename: str,
    show: bool = True,
    output_filename: str = None,
    width: int = 1050,
    cmap: str = "viridis",
    alpha: float = 0.65,
    extended: bool = True,
    periods_remove: List[int] = None,
    groups_remove: List[int] = None,
    log_scale: bool = False,
    cbar_height: float = None,
    cbar_standoff: int = 12,
    cbar_fontsize: int = 14,
    blank_color: str = "#c4c4c4",
    under_value: float = None,
    under_color: str = "#140F0E",
    over_value: float = None,
    over_color: str = "#140F0E",
    ox_states : list = [],
    special_elements: List[str] = None,
    special_color: str = "#6F3023",
) -> figure:

    """
    Plot a heatmap over the periodic table of elements.

    Parameters
    ----------
    filename : str
        Path to the .csv file containing the data to be plotted.
    show : str
        If True, the plot will be shown.
    output_filename : str
        If not None, the plot will be saved to the specified (.html) file.
    width : float
        Width of the plot.
    cmap : str
        plasma, inferno, viridis, magma, cividis, turbo
    alpha : float
        Alpha value (transparency).
    extended : bool
        If True, the lanthanoids and actinoids will be shown.
    periods_remove : List[int]
        Period numbers to be removed from the plot.
    groups_remove : List[int]
        Group numbers to be removed from the plot.
    log_scale : bool
        If True, the colorbar will be logarithmic.
    cbar_height : int
        Height of the colorbar.
    cbar_standoff : int
        Distance between the colorbar and the plot.
    cbar_fontsize : int
        Fontsize of the colorbar label.
    blank_color : str
        Hexadecimal color of the elements without data.
    under_value : float
        Values <= under_value will be colored with under_color.
    under_color : str
        Hexadecimal color to be used for the lower bound color.
    over_value : float
        Values >= over_value will be colored with over_color.
    under_color : str
        Hexadecial color to be used for the upper bound color.
    special_elements: List[str]
        List of elements to be colored with special_color.
    special_color: str
        Hexadecimal color to be used for the special elements.

    Returns
    -------
    figure
        Bokeh figure object.
    """

    options.mode.chained_assignment = None

    # Assign color palette based on input argument
    if cmap == "plasma":
        cmap = plasma
        bokeh_palette = "Plasma256"
    elif cmap == "inferno":
        cmap = inferno
        bokeh_palette = "Inferno256"
    elif cmap == "magma":
        cmap = magma
        bokeh_palette = "Magma256"
    elif cmap == "viridis":
        cmap = viridis
        bokeh_palette = "Viridis256"
    elif cmap == "cividis":
        cmap = cividis
        bokeh_palette = "Cividis256"
    elif cmap == "turbo":
        cmap = turbo
        bokeh_palette = "Turbo256"
    else:
        ValueError("Invalid color map.")

    # Define number of and groups
    period_label = ["1", "2", "3", "4", "5", "6", "7"]
    group_range = [str(x) for x in range(1, 19)]

    # Remove any groups or periods
    if groups_remove:
        for gr in groups_remove:
            gr = gr.strip()
            group_range.remove(str(gr))
    if periods_remove:
        for pr in periods_remove:
            pr = pr.strip()
            period_label.remove(str(pr))

    # Read in data from CSV file
    data_elements = []
    data_list = []
    for row in reader(open(filename)):
        data_elements.append(row[0])
        data_list.append(row[1])
    data = [float(i) for i in data_list]

    if len(data) != len(data_elements):
        raise ValueError("Unequal number of atomic elements and data points")

    period_label.append("blank")
    period_label.append("La")
    period_label.append("Ac")

    if extended:
        count = 0
        for i in range(56, 70):
            elements.period[i] = "La"
            elements.group[i] = str(count + 4)
            count += 1

        count = 0
        for i in range(88, 102):
            elements.period[i] = "Ac"
            elements.group[i] = str(count + 4)
            count += 1

    # Define matplotlib and bokeh color map
    if log_scale:
        for datum in data:
            if datum < 0:
                raise ValueError(
                    f"Entry for element {datum} is negative but log-scale is selected"
                )
        color_mapper = LogColorMapper(
            palette=bokeh_palette, low=min(data), high=max(data)
        )
        norm = LogNorm(vmin=min(data), vmax=max(data))
    else:
        color_mapper = LinearColorMapper(
            palette=bokeh_palette, low=min(data), high=max(data)
        )
        norm = Normalize(vmin=min(data), vmax=max(data))
    color_scale = ScalarMappable(norm=norm, cmap=cmap).to_rgba(data, alpha=None)

    # Set blank color
    color_list = [blank_color] * len(elements)

    # Compare elements in dataset with elements in periodic table
    for i, data_element in enumerate(data_elements):
        element_entry = elements.symbol[
            elements.symbol.str.lower() == data_element.lower()
        ]
        if element_entry.empty == False:
            element_index = element_entry.index[0]
        else:
            warnings.warn("Invalid chemical symbol: " + data_element)
        if color_list[element_index] != blank_color:
            warnings.warn("Multiple entries for element " + data_element)
        elif under_value is not None and data[i] <= under_value:
            color_list[element_index] = under_color
        elif over_value is not None and data[i] >= over_value:
            color_list[element_index] = over_color
        else:
            color_list[element_index] = to_hex(color_scale[i])

    if special_elements:
        for k, v in elements["symbol"].iteritems():
            if v in special_elements:
                color_list[k] = special_color

    # Define figure properties for visualizing data
    source = ColumnDataSource(
        data=dict(
            group=[str(x) for x in elements["group"]],
            period=[str(y) for y in elements["period"]],
            sym=elements["symbol"],
            atomic_number=elements["atomic number"],
            type_color=color_list,
        )
    )

    # Plot the periodic table
    p = figure(x_range=group_range, y_range=list(reversed(period_label)), tools="save")
    p.width = width
    p.outline_line_color = None
    p.background_fill_color = None
    p.border_fill_color = None
    p.toolbar_location = "above"
    p.rect("group", "period", 0.9, 0.9, source=source, alpha=alpha, color="type_color")
    p.axis.visible = False
    text_props = {
        "source": source,
        "angle": 0,
        "color": "black",
        "text_align": "left",
        "text_baseline": "middle",
    }
    x = dodge("group", -0.4, range=p.x_range)
    y = dodge("period", 0.3, range=p.y_range)
    y2 = dodge("period", -0.3, range=p.y_range)
    p.text(
        x=x,
        y="period",
        text="sym",
        text_font_style="bold",
        text_font_size="16pt",
        **text_props,
    )
    p.text(x=x, y=y, text="atomic_number", text_font_size="11pt", **text_props)


    color_bar = ColorBar(
        color_mapper=color_mapper,
        ticker=LogTicker(desired_num_ticks=10),
        border_line_color=None,
        label_standoff=cbar_standoff,
        location=(0, 0),
        orientation="vertical",
        scale_alpha=alpha,
        major_label_text_font_size=f"{cbar_fontsize}pt",
    )


    if cbar_height is not None:
        color_bar.height = cbar_height

    p.add_layout(color_bar, "right")
    p.grid.grid_line_color = None

    if output_filename:
        output_file(output_filename)

    if show:
        show_(p)

    return p


In [5]:
df = pd.read_pickle('full_df.pkl')

In [ ]:
df.columns

In [ ]:
df.iloc[0]['numbers']

In [10]:
atom_counts = {}
n_atoms = []
for i,row in df.iterrows():
    symbols = [io_ptable.elements[x] for x in row['numbers']]
    n_atoms.append(len(symbols))
    syms,sym_count = np.unique(symbols,return_counts=True)
    for j,sym in enumerate(syms):
        atom_counts[sym] = atom_counts.get(sym,0) + sym_count[j]
df['n_atoms'] = n_atoms

In [11]:
with open('counts_v1.csv','w') as file1:
    for i,e in enumerate(io_ptable.elements[1:]):
        if atom_counts.get(e,0) > 0:
            file1.write('{},{}\n'.format(e,atom_counts.get(e,0)))

In [ ]:
out = ptable_plotter('counts_v1.csv',
                     # output_filename='example.png',
                     show=True,
                     log_scale=True,
                     periods_remove=['7'],
                     alpha=0.65)

In [18]:
csd_df = pd.read_csv('../1_CSD_compatable_structures.csv')

In [ ]:
csd_df.columns

In [ ]:
import matplotlib.pyplot as plt
df.n_atoms.hist(bins=np.arange(0,250,10),color='b',alpha=0.5,label='IMS Training Set')
csd_df.natoms.hist(bins=np.arange(0,250,10),color='r',alpha=0.5, label='CSD Dataset')
plt.xlabel('Number of Atoms')
plt.ylabel('Counts')
plt.legend()

In [ ]:
csd_df.chemical_formula

In [30]:
atom_counts = {}
n_atoms = []
for i,row in csd_df.iterrows():
    form = row['chemical_formula'].replace('-','').replace('+','').split()
    symbols = []
    sym_count = []
    for thing in form:
        if len(thing) > 1:
            tsym = ''
            tcount = ''
            for item in thing:
                if item.isnumeric():
                    tcount += item
                else:
                    tsym += item
            symbols.append(tsym)
            sym_count.append(int(tcount))
    for j,sym in enumerate(symbols):
        atom_counts[sym] = atom_counts.get(sym,0) + sym_count[j]

In [31]:
with open('counts_csd_v1.csv','w') as file1:
    for i,e in enumerate(io_ptable.elements[1:]):
        if atom_counts.get(e,0) > 0:
            file1.write('{},{}\n'.format(e,atom_counts.get(e,0)))

In [41]:
df['charge'] = [np.sum(row['initial_charges']) for i,row in df.iterrows()]

In [ ]:
df.columns

In [ ]:
len(df.architector_run_label.value_counts())

In [ ]:
len(df[df.relaxed])

In [ ]:
df.charge.hist(bins=np.arange(-2,5,1),color='b',alpha=0.5,label='IMS Training Set',align='left')
csd_df.user_labeled_charge.hist(bins=np.arange(-2,5,1),color='r',alpha=0.5, label='CSD Dataset',align='left')
plt.xlabel('Complex Charge')
plt.ylabel('Counts')
plt.legend()

In [ ]:
view_structures(df.sample(4,random_state=42).mol2string,columns=2)